<a href="https://colab.research.google.com/github/Adarsh3589/AMS5659-15-5PH-BOEING/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile
import requests
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm, anderson

from google.colab import files
from IPython.display import display
import ipywidgets as widgets


# Step 1: Download the Excel file from GitHub
url = "https://raw.githubusercontent.com/Adarsh3589/AMS5659-15-5PH-DATA-/ae94dc3642843476b37acae8a9a468726a753ae4/Data%2015-5%20PH.xlsx"

response = requests.get(url)
response.raise_for_status()
excel_data = BytesIO(response.content)

# Step 2: Load the second sheet
df = pd.read_excel(excel_data, sheet_name=0, engine='openpyxl')

# Create output directory
os.makedirs("output", exist_ok=True)

In [2]:
df

,ESR,Melting Date,Grade,Customer,Size,C,Mn,Si,S,p,...,I_L1,I_L2,I_L3,UTS_T,YS_T,EL_T,RA_T,I_T1,I_T2,I_T3
0,701054,2025-04-18,15-5PH,VAL-MET ENGINEERING PVT. LTD.,79.4 DIA,0.049,0.94,0.43,0.0030,0.019,...,NaN,NaN,NaN,1110.26,1090.56,17.56,58.63,NaN,NaN,NaN
1,700496,2022-09-22,15-5PH,WIPRO ENTERPRISES (P) LTD,95 DIA,0.039,0.89,0.44,0.0030,0.017,...,NaN,NaN,NaN,1247.44,1222.00,10.00,53.32,NaN,NaN,NaN
2,700497,2022-09-23,17-4PH,WIPRO ENTERPRISES (P) LTD,95 DIA,0.037,0.90,0.44,0.0010,0.017,...,NaN,NaN,NaN,1242.56,1208.52,10.60,49.82,NaN,NaN,NaN
3,700586,2023-01-14,17-4PH,VAL-MET ENGINEERING PVT. LTD.,95 DIA,0.052,0.82,0.41,0.0042,0.014,...,NaN,NaN,NaN,1149.80,1068.10,15.50,54.60,NaN,NaN,NaN
4,700590,2023-01-20,17-4PH,"MTAR TECHNOLOGIES LTD, UNIT-II",101 DIA,0.053,0.82,0.38,0.0042,0.013,...,NaN,NaN,NaN,1150.30,1080.30,17.20,54.70,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,701137,2025-06-09,17-4PH,AVERY INDIA LIMITED,WIP,0.045,0.84,0.42,0.0030,0.019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
277,701138,2025-06-09,17-4PH,AVERY INDIA LIMITED,WIP,0.041,0.86,0.44,0.0040,0.019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278,701139,2025-06-18,17-4PH,AVERY INDIA LIMITED,WIP,0.043,0.84,0.42,0.0030,0.019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
279,701140,2025-06-10,17-4PH,AVERY INDIA LIMITED,WIP,0.040,0.85,0.42,0.0040,0.018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.columns

Index(['ESR', 'Melting Date', 'Grade', 'Customer', 'Size', 'C', 'Mn', 'Si',
       'S', 'p', 'Cr', 'Ni', 'Cu', 'Nb', 'Nb/C ratio', 'HT Type',
       'Ageing_Batch', 'Condition', 'UTS_L', 'YS_L', 'EL_L', 'RA_L', 'SA_1',
       'SA2', 'SA_H', 'PH1', 'PH2', 'PH_H', 'I_L1', 'I_L2', 'I_L3', 'UTS_T',
       'YS_T', 'EL_T', 'RA_T', 'I_T1', 'I_T2', 'I_T3'],
      dtype='object')

In [4]:
# Cell 2: Drop rows with missing 'Condition' or 'HT Type'
df.dropna(subset=['Condition', 'HT Type'], inplace=True)
print("Rows after dropping NA in Condition or HT Type:", df.shape)


Rows after dropping NA in Condition or HT Type: (245, 38)


In [5]:
# Cell 3: Drop specified columns if they exist
cols_to_drop = ['SA_1','SA2','PH1','PH2','Ageing_Batch']  # Example list

for col in cols_to_drop:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)
        print(f"Dropped column: {col}")
    else:
        print(f"Column '{col}' not found, skipping...")

print("Columns after dropping:", df.columns.tolist())


Dropped column: SA_1
Dropped column: SA2
Dropped column: PH1
Dropped column: PH2
Dropped column: Ageing_Batch
Columns after dropping: ['ESR', 'Melting Date', 'Grade', 'Customer', 'Size', 'C', 'Mn', 'Si', 'S', 'p', 'Cr', 'Ni', 'Cu', 'Nb', 'Nb/C ratio', 'HT Type', 'Condition', 'UTS_L', 'YS_L', 'EL_L', 'RA_L', 'SA_H', 'PH_H', 'I_L1', 'I_L2', 'I_L3', 'UTS_T', 'YS_T', 'EL_T', 'RA_T', 'I_T1', 'I_T2', 'I_T3']


In [6]:
df.columns

Index(['ESR', 'Melting Date', 'Grade', 'Customer', 'Size', 'C', 'Mn', 'Si',
       'S', 'p', 'Cr', 'Ni', 'Cu', 'Nb', 'Nb/C ratio', 'HT Type', 'Condition',
       'UTS_L', 'YS_L', 'EL_L', 'RA_L', 'SA_H', 'PH_H', 'I_L1', 'I_L2', 'I_L3',
       'UTS_T', 'YS_T', 'EL_T', 'RA_T', 'I_T1', 'I_T2', 'I_T3'],
      dtype='object')

In [9]:
# Cell 4: Dropdown for Condition + Multi-select for Size
import ipywidgets as widgets
from IPython.display import display

# === Step 1: Unique conditions for dropdown ===
unique_conditions = sorted(df['Condition'].dropna().unique())
condition_dropdown = widgets.Dropdown(
    options=unique_conditions,
    value="H1025" if "H1025" in unique_conditions else unique_conditions[0],
    description="Condition:",
    disabled=False
)

# === Step 2: Unique sizes for multi-select ===
unique_sizes = sorted(df['Size'].dropna().unique())
size_options = ["All"] + list(unique_sizes)
size_multiselect = widgets.SelectMultiple(
    options=size_options,
    value=("All",),  # Default = All
    description="Size:",
    disabled=False
)

# === Step 3: Function to filter data ===
def filter_data(condition_value, size_values):
    filtered_df = df[df['Condition'] == condition_value]
    filtered_df = filtered_df.sort_values(by=['ESR','Melting Date'], ascending=True)
    if "All" not in size_values:
        filtered_df = filtered_df[filtered_df['Size'].isin(size_values)]
    global filtered_df
    print(f"Filtered data shape: {filtered_df.shape}")
    display(filtered_df.head())

# === Step 4: Interactive widget to filter on change ===
ui = widgets.VBox([condition_dropdown, size_multiselect])
out = widgets.interactive_output(
    filter_data,
    {"condition_value": condition_dropdown, "size_values": size_multiselect}
)

display(ui, out)


SyntaxError: name 'filtered_df' is used prior to global declaration (ipython-input-2007644633.py, line 30)

In [8]:
df_filtered

NameError: name 'df_filtered' is not defined